# 🌍 Clustering de Países com Dados GEM
## Comparativo: Brasil vs. Mundo

**Objetivo:** Identificar grupos de países com perfis empreendedores similares usando dados no padrão do *Global Entrepreneurship Monitor (GEM)* e posicionar o Brasil dentro desse mapa global.

**Pipeline:**
1. Geração de dados simulados no padrão GEM APS
2. Análise Exploratória (EDA) com pandas, seaborn e plotly
3. Pré-processamento (normalização, PCA)
4. K-Means (Elbow + Silhouette)
5. Clustering Hierárquico + Dendrograma
6. Interpretação e posicionamento do Brasil

> 💡 **Para usar dados reais:** substitua a Seção 1 pelo CSV do APS disponível em [gemconsortium.org/data](https://www.gemconsortium.org/data/sets?id=aps)


## 0. Imports e configurações

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist

import warnings
warnings.filterwarnings("ignore")

# Config visual
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 130, "figure.figsize": (12, 5)})
SEED = 42
np.random.seed(SEED)

CORES = ["#534AB7", "#1D9E75", "#D85A30", "#D4537E", "#378ADD", "#B5820A"]
print("Ambiente OK ✓")


Ambiente OK ✓


## 1. Dados simulados no padrão GEM APS

Reprodução dos principais indicadores do *Adult Population Survey* agregados por país.
Os perfis por região foram calibrados com base nos relatórios GEM 2019–2024.

| Variável | Descrição |
|---|---|
| `tea` | Total Early-Stage Entrepreneurial Activity (%) |
| `estab` | Taxa de empreendedores estabelecidos (%) |
| `intencao` | Intenção de empreender nos próximos 3 anos (%) |
| `oportunidade` | % que empreende por oportunidade (vs. necessidade) |
| `medo_fracasso` | % com medo de fracassar |
| `redes` | % que conhece empreendedor recente |
| `inovacao` | % com produto/serviço novo para o mercado |
| `tech_uso` | % com uso de tecnologias recentes no negócio |
| `regiao` | Grupo geográfico / econômico |


In [2]:
def perfil(media, std, n, lo=2, hi=80):
    return np.clip(np.random.normal(media, std, n), lo, hi)

regioes = {
    "América Latina": {
        "paises": ["Brazil","Colombia","Chile","Mexico","Argentina",
                   "Peru","Ecuador","Guatemala","Panama","Uruguay"],
        "tea": (25,5), "estab": (12,3), "intencao": (42,8),
        "oportunidade": (54,10), "medo_fracasso": (51,8), "redes": (53,8),
        "inovacao": (30,8), "tech_uso": (27,7),
    },
    "Europa Ocidental": {
        "paises": ["Germany","France","Netherlands","Sweden","Spain",
                   "Italy","Finland","Norway","Denmark","Austria"],
        "tea": (9,2), "estab": (10,2), "intencao": (15,4),
        "oportunidade": (76,6), "medo_fracasso": (39,7), "redes": (37,6),
        "inovacao": (50,8), "tech_uso": (52,7),
    },
    "Europa Oriental": {
        "paises": ["Poland","Czech Republic","Hungary","Romania","Croatia"],
        "tea": (11,3), "estab": (9,2), "intencao": (17,4),
        "oportunidade": (68,8), "medo_fracasso": (44,7), "redes": (40,6),
        "inovacao": (42,7), "tech_uso": (44,7),
    },
    "Anglo-saxônica": {
        "paises": ["United States","Canada","United Kingdom","Australia","Ireland"],
        "tea": (15,3), "estab": (12,2), "intencao": (21,4),
        "oportunidade": (78,5), "medo_fracasso": (39,6), "redes": (44,6),
        "inovacao": (55,7), "tech_uso": (59,6),
    },
    "Ásia Desenvolvida": {
        "paises": ["Japan","South Korea","Taiwan","Singapore"],
        "tea": (8,2), "estab": (8,2), "intencao": (13,4),
        "oportunidade": (70,8), "medo_fracasso": (48,7), "redes": (28,5),
        "inovacao": (42,8), "tech_uso": (52,8),
    },
    "Ásia Emergente": {
        "paises": ["China","India","Indonesia","Malaysia","Vietnam"],
        "tea": (13,4), "estab": (10,2), "intencao": (20,5),
        "oportunidade": (63,9), "medo_fracasso": (44,7), "redes": (36,7),
        "inovacao": (47,9), "tech_uso": (48,9),
    },
    "África / Oriente Médio": {
        "paises": ["South Africa","Egypt","Saudi Arabia","Nigeria","Morocco"],
        "tea": (22,6), "estab": (9,2), "intencao": (28,7),
        "oportunidade": (43,10), "medo_fracasso": (48,8), "redes": (46,8),
        "inovacao": (26,7), "tech_uso": (20,7),
    },
}

variaveis = ["tea","estab","intencao","oportunidade","medo_fracasso","redes","inovacao","tech_uso"]
rows = []
for regiao, cfg in regioes.items():
    n = len(cfg["paises"])
    for i, pais in enumerate(cfg["paises"]):
        row = {"country": pais, "regiao": regiao}
        for v in variaveis:
            row[v] = float(perfil(*cfg[v], 1)[0])
        rows.append(row)

df = pd.DataFrame(rows)
df["is_brazil"] = df["country"] == "Brazil"

print(f"Dataset: {df.shape[0]} países, {len(variaveis)} variáveis")
df.head()


Dataset: 44 países, 8 variáveis


,country,regiao,tea,estab,intencao,oportunidade,medo_fracasso,redes,inovacao,tech_uso,is_brazil
0,Brazil,América Latina,27.483571,11.585207,47.181508,69.230299,49.126773,51.126904,42.633703,32.372043,True
1,Colombia,América Latina,22.652628,13.627680,38.292658,49.342702,52.935698,37.693758,16.200657,23.063987,False
2,Chile,América Latina,19.935844,12.942742,34.735807,39.876963,62.725190,51.193790,30.540226,17.026763,False
3,Mexico,América Latina,22.278086,12.332768,32.792051,57.756980,46.194890,50.666450,25.186347,39.965947,False
4,Argentina,América Latina,24.932514,8.826867,48.580359,41.791564,52.670909,37.322639,19.374512,28.378029,False
